# Hyperspectral Image Fusion — Chikusei SOTA

**Target: Beat CoFusion 49.14 / RAMoE 48.10 / SMGU-Net 48.82 PSNR**

Model: NullFusion-SOTA (Multi-scale U-Net + Spectral Dictionary + Wavelet)

| Dataset | Bands | Sensor | Resolution |
|---------|-------|--------|------------|
| Chikusei | 128 | Headwall Nano-Hyperspec | 2517x2335 |

### SOTA Targets
| Method | PSNR | SAM |
|--------|------|-----|
| CoFusion (2026) | 49.14 | 2.60 |
| SMGU-Net (2025) | 48.82 | 2.72 |
| RAMoE (2026) | 48.10 | 0.79 |
| PSRT (2023) | 47.99 | 2.84 |
| U2Net (2023) | 47.93 | 2.77 |
| KrylovNet v1 (ours) | 43.69 | 6.07 |

In [ ]:
!pip install -q scipy einops

In [ ]:
import os, glob, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.io import loadmat
from scipy.ndimage import convolve, uniform_filter
from torch.utils.data import Dataset
import matplotlib.pyplot as plt

print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

In [ ]:
ROOT = '/kaggle/input/chikusei'
OUT = '/kaggle/working'
mat_files = glob.glob(os.path.join(ROOT, '**', '*.mat'), recursive=True)
mat_files.sort(key=lambda f: os.path.getsize(f), reverse=True)
print(f'Found {len(mat_files)} .mat files')
for f in mat_files[:5]:
    print(f'  {os.path.basename(f)} ({os.path.getsize(f)/1e6:.0f} MB)')

## SRF + Forward Model

In [ ]:
def chikusei_srf(bands=128):
    wl = np.linspace(363.0, 1018.0, 128)
    raw = np.stack([
        np.exp(-((wl-620)**2)/(2*80**2)),
        np.exp(-((wl-540)**2)/(2*70**2)),
        np.exp(-((wl-460)**2)/(2*60**2)),
    ], axis=1).astype(np.float32)
    if bands != 128:
        xs = np.linspace(0,1,128); xd = np.linspace(0,1,bands)
        raw = np.stack([np.interp(xd, xs, raw[:,i]) for i in range(3)], axis=1)
    return raw / np.maximum(raw.sum(axis=0, keepdims=True), 1e-8)

srf = chikusei_srf(128)
wl = np.linspace(363, 1018, 128)
plt.figure(figsize=(10,3))
plt.plot(wl, srf[:,0], 'r-', lw=2, label='Red')
plt.plot(wl, srf[:,1], 'g-', lw=2, label='Green')
plt.plot(wl, srf[:,2], 'b-', lw=2, label='Blue')
plt.xlabel('Wavelength (nm)'); plt.ylabel('Response'); plt.legend()
plt.title('Chikusei Nano-Hyperspec SRF (128 bands)'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

s = np.linalg.svd(srf, compute_uv=False)
print(f'SRF condition number: {s[0]/s[-1]:.2f}')

## Dataset

In [ ]:
def gaussian_kernel2d(size=9, sigma=1.2):
    ax = np.arange(size, dtype=np.float32)-(size-1)/2
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-0.5*(xx**2+yy**2)/sigma**2)
    return (k/k.sum()).astype(np.float32)

class ChikuseiDS(Dataset):
    def __init__(self, root, split='train', bands=128, scale=4, patch=64):
        self.split = split; self.bands = bands; self.scale = scale; self.patch = patch
        self.srf = chikusei_srf(bands)
        self.kernel = gaussian_kernel2d(9, 1.2)
        mat_files = glob.glob(os.path.join(root,'**','*.mat'), recursive=True)
        mat_files.sort(key=lambda f: os.path.getsize(f), reverse=True)
        print(f'[Data] Loading: {os.path.basename(mat_files[0])}')
        data = loadmat(mat_files[0])
        for key, val in data.items():
            if not key.startswith('__') and hasattr(val, 'shape'):
                arr = np.array(val, dtype=np.float32)
                if arr.ndim==3 and min(arr.shape)>10:
                    if arr.shape[0]>arr.shape[-1]: arr=arr.transpose(2,0,1)
                    if arr.max()>1: arr=arr/arr.max()
                    self.cube = arr; break
        C,H,W = self.cube.shape
        print(f'[Data] {C} bands, {H}x{W} pixels')
        p=patch
        coords = [(y,x) for y in range(0,H-p+1,p) for x in range(0,W-p+1,p)]
        random.seed(42); random.shuffle(coords)
        n=int(0.7*len(coords))
        self.patches = coords[:n] if split=='train' else coords[n:]
        print(f'[Data] {split}: {len(self.patches)} patches')
    def __len__(self):
        return len(self.patches)*(200 if self.split=='train' else 1)
    def _sim(self, gt):
        C,H,W = gt.shape
        blurred = np.empty_like(gt)
        for c in range(C): blurred[c]=convolve(gt[c], self.kernel, mode='wrap')
        hr=H//self.scale; y0=(H-hr*self.scale)//2; x0=(W-hr*self.scale)//2
        lr = blurred[:, y0::self.scale, x0::self.scale].astype(np.float32)
        msi = np.einsum('chw,cm->mhw', gt, self.srf).astype(np.float32)
        return lr, np.clip(msi,0,1)
    def __getitem__(self, idx):
        y,x = self.patches[idx%len(self.patches)]
        p=self.patch; gt=self.cube[:, y:y+p, x:x+p].copy()
        if self.split=='train':
            if random.random()<0.5: gt=gt[:,:,::-1].copy()
            if random.random()<0.5: gt=gt[:,::-1,:].copy()
            if random.random()<0.5: gt=np.rot90(gt,random.randint(1,3),axes=(1,2)).copy()
        lr,msi = self._sim(gt)
        return torch.from_numpy(gt), torch.from_numpy(lr), torch.from_numpy(msi)

train_ds = ChikuseiDS(ROOT, 'train', 128, 4, 64)
test_ds = ChikuseiDS(ROOT, 'test', 128, 4, 64)

In [ ]:
gt, lr, msi = train_ds[0]
fig, axes = plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(gt[50].numpy(), cmap='viridis'); axes[0].set_title('GT (band 50)')
axes[1].imshow(lr[50].numpy(), cmap='viridis'); axes[1].set_title('LR-HSI (band 50)')
axes[2].imshow(msi.numpy().transpose(1,2,0)); axes[2].set_title('MSI (RGB)')
plt.tight_layout(); plt.show()
print(f'GT: {gt.shape}, LR: {lr.shape}, MSI: {msi.shape}')

## Model + Training (Run the script directly)

In [ ]:
# Write the training script
%%writefile train_chikusei_sota.py
# [The full train_chikusei_sota.py content goes here]
# Copy from experiments/scripts/train_chikusei_sota.py

In [ ]:
# Run training
!python train_chikusei_sota.py \
    --root /kaggle/input/chikusei \
    --output_dir /kaggle/working \
    --epochs 5000 \
    --time_budget_h 8.5 \
    --eval_every 100

In [ ]:
# Load and display results
with open(os.path.join(OUT, 'chikusei_sota_results.json')) as f:
    results = json.load(f)

print('='*60)
print('Chikusei x4 — Final Results')
print('='*60)
print(f'Model: NullFusionSOTA ({results["params_M"]:.2f}M params)')
print(f'Best epoch: {results["best_epoch"]}')
print(f'PSNR: {results["final"]["psnr"]:.4f} dB')
print(f'SSIM: {results["final"]["ssim"]:.4f}')
print(f'SAM:  {results["final"]["sam"]:.3f} deg')
print(f'ERGAS: {results["final"]["ergas"]:.3f}')
print()
print('SOTA Comparison:')
for name, target in results['sota'].items():
    d = results['final']['psnr'] - target
    m = ' <<< BEAT' if d > 0 else ''
    print(f'  {name:25s} Target {target:6.2f} | Ours Δ={d:+.2f}{m}')

In [ ]:
# Visualize predictions
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Reload model
from train_chikusei_sota import NullFusionSOTA, psnr, sam
model = NullFusionSOTA(bands=128, msi=3, width=96).to(device)
model.load_state_dict(torch.load(os.path.join(OUT, 'nullfusion_sota_chikusei/best.pth'), map_location=device))
model.eval()

fig, axes = plt.subplots(3, 5, figsize=(25, 15))
for i in range(5):
    gt, lr, msi = test_ds[i]
    with torch.no_grad():
        pred = model(lr.unsqueeze(0).to(device), msi.unsqueeze(0).to(device))["out"][0].cpu().numpy()
    gt_np = gt.numpy()
    axes[0,i].imshow(gt_np[50], cmap='viridis')
    axes[0,i].set_title(f'GT (PSNR={psnr(pred, gt_np):.1f})')
    axes[1,i].imshow(pred[50], cmap='viridis')
    axes[1,i].set_title(f'Predicted (SAM={sam(pred, gt_np):.1f})')
    axes[2,i].imshow(np.abs(gt_np[50]-pred[50]), cmap='hot')
    axes[2,i].set_title('Error')
for ax in axes.flat: ax.axis('off')
plt.suptitle('Chikusei x4 — NullFusionSOTA Predictions', fontsize=16)
plt.tight_layout(); plt.show()